In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

In [2]:
def select_pca(indep_X,n_components):
    pca = PCA(n_components=n_components)
    pca_features = pca.fit_transform(indep_X)
    return pca_features
    
def split_scalar(indep_X, dep_Y):
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size=0.25, random_state=0)
    
    # Scale the features
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test) 
    # Use transform, not fit_transform, on the test set
    
    return X_train, X_test, y_train, y_test

 
def cm_prediction(classifier,X_test):
     y_pred = classifier.predict(X_test)
        
        # Making the Confusion Matrix
     from sklearn.metrics import confusion_matrix
     cm = confusion_matrix(y_test, y_pred)
        
     from sklearn.metrics import accuracy_score 
     from sklearn.metrics import classification_report 
        #from sklearn.metrics import confusion_matrix
        #cm = confusion_matrix(y_test, y_pred)
        
     Accuracy=accuracy_score(y_test, y_pred )
        
     report=classification_report(y_test, y_pred)
     return  classifier,Accuracy,report,X_test,y_test,cm

def logistic(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      
    
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
    
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
   
def Navie(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm         
    
    
def knn(X_train,y_train,X_test):
           
        # Fitting K-NN to the Training set
        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
def Decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training set
        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      


def random(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
 
def selectk_Classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf):

    dataframe = pd.DataFrame({
        'Logistic': acclog,
        'SVMl': accsvml,
        'SVMnl': accsvmnl,
        'KNN': accknn,
        'Navie': accnav,
        'Decision': accdes,
        'Random': accrf
    }, index=['PCA'])

    return dataframe


In [3]:
dataset1 = pd.read_csv("Elevator_predictive_maintenance_cleaned.csv", index_col=None)

# Create a copy of the dataset
df2 = dataset1.copy()

# Convert categorical variables to dummy/indicator variables
df2 = pd.get_dummies(df2, dtype=int ,drop_first=True)

# Separate the independent and dependent variables
indep_X = df2.drop('Maintenance_Required', axis=1)  # Drop the dependent variable column
dep_Y = df2['Maintenance_Required']                  # Set 'Customer_Segment' as the dependent variable
dep_Y=dep_Y.astype(int)

In [4]:
df2

,Motor_Temperature_C,Motor_Vibration_mm_s,Motor_Current_A,Motor_Speed_RPM,Brake_Temperature_C,Door_Opening_Time_s,Door_Closing_Time_s,Load_Percentage,Number_of_Trips,Operating_Hours,Error_Count,Days_Since_Maintenance,Previous_Faults,Maintenance_Required
0,94.56,1.84,20.15,1470.3,57.150000,2.91,3.09,70.42,83.000000,2889.6,5.0,224.3,3.0,0.0
1,105.00,4.27,20.10,1420.8,84.050000,3.83,4.28,49.96,192.000000,7276.7,13.0,602.8,7.0,1.0
2,67.60,1.64,19.80,1471.1,46.360000,2.23,2.96,80.75,31.000000,1082.3,1.0,101.4,1.0,0.0
3,105.00,4.24,13.63,1443.8,80.730000,3.33,3.75,29.09,111.492723,6716.5,12.0,575.6,9.0,1.0
4,92.52,2.13,18.64,1466.5,62.520000,3.11,3.06,56.64,105.000000,3116.7,4.0,281.1,2.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
480,75.25,2.01,18.61,1488.1,44.030000,2.26,2.65,72.45,36.000000,1156.6,2.0,76.5,1.0,0.0
481,99.81,2.98,16.39,1456.0,54.790000,3.34,2.72,74.99,40.000000,2621.0,7.0,173.2,2.0,0.0
482,99.31,2.66,19.45,1443.8,65.636737,2.67,2.94,47.38,76.000000,3639.4,4.0,256.4,2.0,0.0
483,85.15,2.41,14.90,1470.0,56.170000,2.73,2.83,40.67,41.000000,2693.9,8.0,218.3,4.0,0.0


In [11]:
pca_features = select_pca(indep_X, 7)  # You only need indep_X and the number of components
       

acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

In [12]:
pca_features

array([[-1.55850772e+03, -2.26444903e+01,  1.08128229e+01, ...,
         6.53816558e-01,  7.59015118e+00, -2.28447417e+00],
       [ 2.84640693e+03,  1.68199015e+01,  1.28602904e+01, ...,
        -8.69517808e+00, -6.14796912e+00,  3.94782388e+00],
       [-3.37071939e+03, -6.72476457e+00, -3.35800686e-01, ...,
        -1.18943552e+01, -1.11858637e+01, -2.99360791e-02],
       ...,
       [-8.08583354e+02, -5.02849546e+01, -9.37598085e+00, ...,
        -2.31030878e+01,  8.69433694e+00,  3.14434958e+00],
       [-1.75504439e+03, -1.83165824e+01, -2.81194042e+01, ...,
        -3.76154851e+00,  1.28963968e+00,  2.42310168e+00],
       [-1.45493419e+03, -1.31932685e+00, -1.15954577e+01, ...,
         1.08544812e+01,  1.19677010e+00,  2.48192790e+00]],
      shape=(485, 7))

In [13]:
import warnings
warnings.filterwarnings("ignore")

In [14]:
X_train, X_test, y_train, y_test = split_scalar(pca_features,dep_Y)
   
    
        
classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
acclog.append(Accuracy)

classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
accsvml.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
accsvmnl.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
accknn.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test)  
accnav.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
accdes.append(Accuracy)
    
classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
accrf.append(Accuracy)
    
 


In [15]:
result =selectk_Classification (
    acclog,
    accsvml,
    accsvmnl,
    accknn,
    accnav,
    accdes,
    accrf
)

In [10]:
result

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
PCA,0.803279,0.795082,0.786885,0.795082,0.786885,0.721311,0.770492


In [16]:
result#7

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
PCA,0.795082,0.819672,0.778689,0.786885,0.778689,0.729508,0.786885
